<a href="https://colab.research.google.com/github/lidia-azizian/prompt-engineering/blob/main/01_conversation_management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Conversation Management with Gemini

Exploring how LLMs maintain context across multiple interactions using Python and the Gemini API.

This notebook covers:
1. Predefined conversation history
2. Manually updating conversation history
3. Server-side conversation chaining

**Technologies:** Python, Google Colab, Gemini API, OpenAI-compatible SDK, Google Gen AI SDK.

**Reference:** Inspired by [The Complete Prompt Engineering for AI Bootcamp (2026)](https://github.com/BrightPool/udemy-prompt-engineering-course). Examples adapted for Gemini with additional explanations.


In [1]:

%pip install -q -U openai

from openai import OpenAI
import getpass

MODEL = "gemini-3.8-flash"

client = OpenAI(
    api_key=getpass.getpass("Enter your Gemini API key: "),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.0 MB/s eta 0:00:00
Enter your Gemini API key: ··········



## 1. Predefined Conversation History

A conversation is represented as a list of dictionaries, where:
- `role` identifies the speaker (`user` or `assistant`).
- `content` contains the message text.

Here, three predefined messages are provided to Gemini, which generates the next response.

**Concept:** Previous messages provide context for interpreting the latest prompt.


In [ ]:

history = [
    {"role": "user", "content": "knock knock."},
    {"role": "assistant", "content": "Who's there?"},
    {"role": "user", "content": "Orange."}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

print("Assistant response:", response.choices[0].message.content)


Assistant response: Orange who?



## 2. Updating Conversation History

This example demonstrates a multi-turn conversation.

After Gemini generates the first joke, its response is added to `history` using `append()`. A new user message is then added, and the updated history is sent in the next API request.

**Concept:** With manual history management, the application stores and resends previous messages to maintain context.


In [ ]:

# Start a new conversation
history = [
    {"role": "user", "content": "tell me a joke"}
]

# First API request
response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

# Print the first joke
first_joke = response.choices[0].message.content
print("Joke:", first_joke)

# Add Gemini's response to conversation history
history.append({
    "role": "assistant",
    "content": first_joke
})

# Add the user's next message
history.append({
    "role": "user",
    "content": "tell me another"
})

# Second API request with updated history
second_response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

print("Another joke:", second_response.choices[0].message.content)


Joke: Why do we tell actors to "break a leg"?

Because every play has a cast!
Another joke: I told my wife she was drawing her eyebrows too high. 

She looked surprised.



## 3. Conversation Chaining

Instead of manually resending conversation history, Gemini's Interactions API can reference an earlier interaction.

The first request generates a response with a unique ID. The next request uses `previous_interaction_id=response.id` to continue the conversation.

**Concept:** Server-side chaining allows the API provider to retrieve previous context using an interaction ID.


In [ ]:
%pip install -q -U google-genai

In [ ]:

from google import genai
import getpass

gemini_client = genai.Client(
    api_key=getpass.getpass("Enter your Gemini API key: ")
)

MODEL = "gemini-3.8-flash"


Enter your Gemini API key: ··········


In [ ]:

# Ask for a joke
response = gemini_client.interactions.create(
    model=MODEL,
    input="tell me a joke"
)

print("Joke:", response.output_text)

# Chain the next request using the previous interaction ID
second_response = gemini_client.interactions.create(
    model=MODEL,
    previous_interaction_id=response.id,
    input="explain why this is funny."
)

print("Explanation:", second_response.output_text)


Joke: Did you hear about the mathematician who's afraid of negative numbers?

He'll stop at nothing to avoid them!
Explanation: The humor in this joke relies on a pun and the double meaning of the phrase **"stop at nothing."**

1. **The Idiom (Figure of Speech):** In everyday language, to "stop at nothing" means you are extremely determined and will do whatever it takes to achieve a goal—in this case, avoiding negative numbers.
2. **The Literal Math Meaning:** On a number line, zero represents "nothing." If you are counting down through positive numbers (3, 2, 1...) toward negative numbers (-1, -2...), you have to pass zero. If you literally stop at zero, you are **stopping at "nothing"** to avoid entering negative territory.

The joke is funny because it cleverly takes a common expression of extreme effort and applies it literally to basic arithmetic.



## Key Takeaways

| Manual History | Conversation Chaining |
|---|---|
| Stores messages in a Python list | References previous interactions by ID |
| Uses `history.append()` | Uses `previous_interaction_id` |
| Resends the conversation history | Provider retrieves previous context |
| Offers direct control over messages | Simplifies follow-up requests |

### Connection to Prompt Engineering

An LLM's response depends not only on the current prompt but also on the context provided.

Managing conversation context is essential for building multi-turn AI applications.

### Attribution
Concepts inspired by *The Complete Prompt Engineering for AI Bootcamp (2026)*. Examples adapted to Google's Gemini API.
